# **Exercise 1: Accuracy vs Number of Simulations**

In [ ]:
import numpy as np
import time
np.random.seed(12345)
# params.
S0 = 60
K = 65
T = 1.0
r = 0.03
sigma = 0.3

# trials
ns = [10000, 20000, 50000, 100000, 200000, 500000]

# Function to simulate stock prices S_T
def simulate_ST(S0, r, sigma, T, n):
    epsilon = np.random.normal(0, 1, n)
    ST = S0 * np.exp((r - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * epsilon)
    return ST

# Function to calculate option prices with percent standard error and runtime
def monte_carlo(S0, K, r, sigma, T, n):
    start_time = time.time()
    ST = simulate_ST(S0, r, sigma, T, n)
    payoff_call = np.maximum(ST - K, 0)
    payoff_put = np.maximum(K - ST, 0)
    discount_call = np.exp(-r * T) * payoff_call
    discount_put = np.exp(-r * T) * payoff_put
    price_call = np.mean(discount_call)
    price_put = np.mean(discount_put)
    se_call = np.std(discount_call, ddof=1) / np.sqrt(n)
    se_put = np.std(discount_put, ddof=1) / np.sqrt(n)
    pct_se_call = (se_call / price_call) * 100
    pct_se_put = (se_put / price_put) * 100
    runtime = time.time() - start_time
    return n, price_call, price_put, pct_se_call, pct_se_put, runtime

results = []
for n in ns:
    results.append(monte_carlo(S0, K, r, sigma, T, n))

# Output
output = []
output.append(f"{'n':>10} {'Call Price':>12} {'Put Price':>12} {'%SE Call':>12} {'%SE Put':>12} {'Runtime (s)':>12}")
for row in results:
    output.append(f"{row[0]:10d} {row[1]:12.4f} {row[2]:12.4f} {row[3]:12.4f} {row[4]:12.4f} {row[5]:12.4f}")

output_str = '\n'.join(output)
print(output_str)

         n   Call Price    Put Price     %SE Call      %SE Put  Runtime (s)
     10000       5.7816       9.0738       1.9433       1.0986       0.0011
     20000       5.9222       9.0039       1.3886       0.7814       0.0015
     50000       5.9041       8.9890       0.8847       0.4945       0.0037
    100000       5.9126       8.9592       0.6234       0.3487       0.0046
    200000       5.9206       8.9537       0.4406       0.2469       0.0090
    500000       5.9117       8.9683       0.2775       0.1564       0.0293


# **Exercise 2: Impact of Extreme Strike Prices on Monte Carlo Accuracy**

In [ ]:
# Parameters
S0 = 60
T = 1.0
r = 0.03
sigma = 0.3
n = 500000  # Use fixed n

# Range of strike prices K
Ks = np.arange(20, 121, 10)

# Function to simulate terminal stock prices S_T
def simulate_ST(S0, r, sigma, T, n):
    epsilon = np.random.normal(0, 1, n)
    ST = S0 * np.exp((r - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * epsilon)
    return ST

# Function to calculate percent standard error for call and put for changing strike K
def monte_carlo(S0, K, r, sigma, T, n):
    ST = simulate_ST(S0, r, sigma, T, n)
    payoff_call = np.maximum(ST - K, 0)
    payoff_put = np.maximum(K - ST, 0)
    discount_call = np.exp(-r * T) * payoff_call
    discount_put = np.exp(-r * T) * payoff_put
    price_call = np.mean(discount_call)
    price_put = np.mean(discount_put)
    se_call = np.std(discount_call, ddof=1) / np.sqrt(n)
    se_put = np.std(discount_put, ddof=1) / np.sqrt(n)
    pct_se_call = (se_call / price_call) * 100 if price_call > 0 else 0
    pct_se_put = (se_put / price_put) * 100 if price_put > 0 else 0
    return K, pct_se_call, pct_se_put

results = []
for K in Ks:
    results.append(monte_carlo(S0, K, r, sigma, T, n))

# Prepare output table
output = []
output.append(f"{'K':>10} {'%SE Call':>12} {'%SE Put':>12}")
for row in results:
    output.append(f"{row[0]:10d} {row[1]:12.4f} {row[2]:12.4f}")

output_str = '\n'.join(output)
print(output_str)

         K     %SE Call      %SE Put
        20       0.0640      16.7315
        30       0.0840       1.6893
        40       0.1159       0.5628
        50       0.1640       0.2938
        60       0.2331       0.1872
        70       0.3318       0.1328
        80       0.4748       0.1000
        90       0.6850       0.0786
       100       0.9846       0.0637
       110       1.4359       0.0530
       120       2.1112       0.0450
